# Stage D: Cross-Sensor Identity Linking & Direction

See `docs/specs/2026-07-30-stage-d-direction-linking-design.md` for the
full design. This notebook: (1) reproduces the three ground-truth
findings that motivated the design, (2) calibrates the matcher against
`DATA/combined_file_manual.csv`, (3) applies it to all dual-sensor dates,
(4) writes the matched-pair output table.

In [1]:
import sys
sys.path.insert(0, ".")
import pandas as pd
import numpy as np
from stage_d_lib import (
    first_session_anchor, last_session_anchor, best_direction_for_pair,
    match_travelers, predicted_counts, calibration_loss, manual_to_internal,
    grid_search, leave_one_date_out,
)

DATA_DIR = "../DATA"

## Load Stage C output and the manual ground truth

In [2]:
sess_a = pd.read_csv(f"{DATA_DIR}/sessions_sensor_A_with_traveler.csv")
sess_b = pd.read_csv(f"{DATA_DIR}/sessions_sensor_B_with_traveler.csv")
for df in (sess_a, sess_b):
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])

manual = pd.read_csv(f"{DATA_DIR}/combined_file_manual.csv")
manual["dt"] = pd.to_datetime(manual["Date"] + " " + manual["Time"], format="%d.%m.%Y %H:%M:%S")
manual["date"] = manual["dt"].dt.date
gt_windows = manual.groupby("date").agg(window_start=("dt", "min"), window_end=("dt", "max")).reset_index()
gt_dirs = manual.groupby(["date", "direction"]).size().unstack(fill_value=0).reset_index()
gt_windows = gt_windows.merge(gt_dirs, on="date")
print(f"Sensor A sessions: {len(sess_a)}, Sensor B sessions: {len(sess_b)}")
print(f"GT dates: {len(gt_windows)}")
gt_windows

Sensor A sessions: 15267, Sensor B sessions: 19359
GT dates: 8


,date,window_start,window_end,A,B
0,2026-04-11,2026-04-11 09:03:54,2026-04-11 09:36:57,1,27
1,2026-05-02,2026-05-02 12:36:26,2026-05-02 13:15:03,36,20
2,2026-05-09,2026-05-09 11:00:35,2026-05-09 11:58:39,56,181
3,2026-05-16,2026-05-16 08:20:51,2026-05-16 09:04:17,2,22
4,2026-05-19,2026-05-19 09:27:20,2026-05-19 10:43:42,2,136
5,2026-05-23,2026-05-23 09:04:50,2026-05-23 09:57:14,13,30
6,2026-05-27,2026-05-27 10:29:41,2026-05-27 10:51:16,0,17
7,2026-05-31,2026-05-31 10:09:30,2026-05-31 10:51:04,1,55


## Ground-truth finding 1: sensors see mostly the same people

Summing each sensor's independent traveler count double-counts, because
most travelers show up on both sensors within the same short window.

In [3]:
def travelers_in_window(sess_df, ws, we):
    win = sess_df[(sess_df["start_time"] >= ws) & (sess_df["start_time"] <= we)]
    return win.groupby("Traveler_ID")["MAC"].apply(set)

overlap_rows = []
for _, r in gt_windows.iterrows():
    ta = travelers_in_window(sess_a, r["window_start"], r["window_end"])
    tb = travelers_in_window(sess_b, r["window_start"], r["window_end"])
    shared_macs = set().union(*ta.values) & set().union(*tb.values) if len(ta) and len(tb) else set()
    overlap_rows.append({
        "date": r["date"], "trav_A": len(ta), "trav_B": len(tb),
        "manual_total": int(r["A"] + r["B"]),
    })
overlap_df = pd.DataFrame(overlap_rows)
print("Sum of A+B traveler counts vs. manual total (should NOT match if double-counting):")
overlap_df

Sum of A+B traveler counts vs. manual total (should NOT match if double-counting):


,date,trav_A,trav_B,manual_total
0,2026-04-11,42,42,28
1,2026-05-02,138,75,56
2,2026-05-09,265,268,237
3,2026-05-16,46,49,24
4,2026-05-19,124,140,138
5,2026-05-23,53,56,43
6,2026-05-27,22,22,17
7,2026-05-31,33,25,56


## Ground-truth finding 2: raw MAC-address matching over-counts

Matching Sensor A and B by literal MAC equality (no per-sensor entity
resolution first) produces a much larger union than the manual total.

In [4]:
a_raw = pd.read_parquet(f"{DATA_DIR}/sensor_A_deduped.parquet")[["MAC", "timestamp"]]
b1 = pd.read_excel(f"{DATA_DIR}/combined_file_sensor_B_part_1.xlsx")
b2 = pd.read_excel(f"{DATA_DIR}/combined_file_sensor_B_part_2.xlsx")
b_raw = pd.concat([b1, b2], ignore_index=True).drop_duplicates(subset=["MAC", "RSSI", "timestamp"])[["MAC", "timestamp"]]

raw_union_rows = []
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    a_macs = set(a_raw[(a_raw["timestamp"] >= ws) & (a_raw["timestamp"] <= we)]["MAC"])
    b_macs = set(b_raw[(b_raw["timestamp"] >= ws) & (b_raw["timestamp"] <= we)]["MAC"])
    union_total = len(a_macs | b_macs)
    manual_total = int(r["A"] + r["B"])
    raw_union_rows.append({
        "date": r["date"], "raw_mac_union": union_total, "manual_total": manual_total,
        "ratio": round(union_total / manual_total, 2) if manual_total else None,
    })
pd.DataFrame(raw_union_rows)

,date,raw_mac_union,manual_total,ratio
0,2026-04-11,64,28,2.29
1,2026-05-02,247,56,4.41
2,2026-05-09,590,237,2.49
3,2026-05-16,64,24,2.67
4,2026-05-19,370,138,2.68
5,2026-05-23,90,43,2.09
6,2026-05-27,37,17,2.18
7,2026-05-31,52,56,0.93


## Ground-truth finding 3: naive "first-seen" order is not a direction signal

Comparing which sensor first detected a shared MAC, against the manual
direction split, shows the opposite skew on some dates -- consistent with
Sensor A having a longer detection range, not true arrival order.

In [5]:
first_seen_rows = []
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    a_first = a_raw[(a_raw["timestamp"] >= ws) & (a_raw["timestamp"] <= we)].groupby("MAC")["timestamp"].min()
    b_first = b_raw[(b_raw["timestamp"] >= ws) & (b_raw["timestamp"] <= we)].groupby("MAC")["timestamp"].min()
    shared = set(a_first.index) & set(b_first.index)
    a_seen_first = sum(1 for m in shared if a_first[m] < b_first[m])
    b_seen_first = sum(1 for m in shared if b_first[m] < a_first[m])
    first_seen_rows.append({
        "date": r["date"], "A_seen_first": a_seen_first, "B_seen_first": b_seen_first,
        "manual_A": int(r["A"]), "manual_B": int(r["B"]),
    })
pd.DataFrame(first_seen_rows)

,date,A_seen_first,B_seen_first,manual_A,manual_B
0,2026-04-11,52,1,1,27
1,2026-05-02,39,60,36,20
2,2026-05-09,341,113,56,181
3,2026-05-16,47,5,2,22
4,2026-05-19,223,26,2,136
5,2026-05-23,38,28,13,30
6,2026-05-27,35,0,0,17
7,2026-05-31,29,4,1,55


## Build per-traveler boundary anchors for each GT window

In [6]:
def build_anchors_for_window(sess_df, ws, we):
    win = sess_df[(sess_df["start_time"] >= ws) & (sess_df["start_time"] <= we)]
    anchors = []
    for tid, grp in win.groupby("Traveler_ID"):
        anchors.append({
            "traveler_id": tid,
            "first": first_session_anchor(grp),
            "last": last_session_anchor(grp),
        })
    return anchors

gt_dates_data = {}
for _, r in gt_windows.iterrows():
    ws, we = r["window_start"], r["window_end"]
    gt_dates_data[str(r["date"])] = {
        "a_anchors": build_anchors_for_window(sess_a, ws, we),
        "b_anchors": build_anchors_for_window(sess_b, ws, we),
        "manual_A": int(r["A"]), "manual_B": int(r["B"]),
    }

for date, d in gt_dates_data.items():
    print(f"{date}: {len(d['a_anchors'])} A travelers, {len(d['b_anchors'])} B travelers, "
          f"manual A={d['manual_A']} B={d['manual_B']}")

2026-04-11: 42 A travelers, 42 B travelers, manual A=1 B=27
2026-05-02: 138 A travelers, 75 B travelers, manual A=36 B=20
2026-05-09: 265 A travelers, 268 B travelers, manual A=56 B=181
2026-05-16: 46 A travelers, 49 B travelers, manual A=2 B=22
2026-05-19: 124 A travelers, 140 B travelers, manual A=2 B=136
2026-05-23: 53 A travelers, 56 B travelers, manual A=13 B=30
2026-05-27: 22 A travelers, 22 B travelers, manual A=0 B=17
2026-05-31: 33 A travelers, 25 B travelers, manual A=1 B=55


## Calibration: determine label orientation, then leave-one-date-out fit

The manual CSV's A/B convention isn't documented, so both orientations
are tried; whichever gives the lower total in-sample loss (across all 8
GT dates, using the best grid params for that orientation) is kept.

In [7]:
import itertools

PARAM_GRID = [
    {"transit_center_s": c, "transit_spread_s": s, "rssi_weight": w, "min_match_score": m}
    for c, s, w, m in itertools.product(
        [15.0, 20.0, 25.0, 30.0, 40.0],
        [10.0, 15.0, 20.0, 30.0],
        [0.5, 1.0, 2.0, 5.0],
        [0.2, 0.4, 0.6],
    )
]
print(f"Grid size: {len(PARAM_GRID)}")

best_orientation = None
best_orientation_loss = float("inf")
for flip in (False, True):
    _, loss = grid_search(gt_dates_data, PARAM_GRID, flip)
    print(f"flip={flip}: best in-sample total loss = {loss:.1f}")
    if loss < best_orientation_loss:
        best_orientation_loss, best_orientation = loss, flip

print(f"\nChosen orientation: flip={best_orientation}")

Grid size: 240


flip=False: best in-sample total loss = 21492.0


flip=True: best in-sample total loss = 19692.0

Chosen orientation: flip=True


In [8]:
loo_rows = leave_one_date_out(gt_dates_data, PARAM_GRID, best_orientation)
loo_report = pd.DataFrame([
    {
        "date": r["date"],
        "predicted_A_to_B": r["predicted"]["A_to_B"], "predicted_B_to_A": r["predicted"]["B_to_A"],
        "predicted_total": r["predicted"]["total"],
        "manual_A_to_B": r["manual"]["A_to_B"], "manual_B_to_A": r["manual"]["B_to_A"],
        "manual_total": r["manual"]["total"],
        "held_out_loss": r["held_out_loss"],
    }
    for r in loo_rows
])
print("Leave-one-date-out held-out predictions vs. ground truth:")
loo_report

Leave-one-date-out held-out predictions vs. ground truth:


,date,predicted_A_to_B,predicted_B_to_A,predicted_total,manual_A_to_B,manual_B_to_A,manual_total,held_out_loss
0,2026-04-11,22,17,39,27,1,28,402
1,2026-05-02,33,42,75,20,36,56,566
2,2026-05-09,134,122,256,181,56,237,6926
3,2026-05-16,21,22,43,22,2,24,762
4,2026-05-19,54,55,109,136,2,138,10374
5,2026-05-23,14,28,42,30,13,43,482
6,2026-05-27,13,8,21,17,0,17,96
7,2026-05-31,15,10,25,55,1,56,2642


In [9]:
final_params, final_loss = grid_search(gt_dates_data, PARAM_GRID, best_orientation)
print(f"Final production params (fit on all 8 GT dates): {final_params}")
print(f"In-sample total loss: {final_loss:.1f}  |  mean held-out loss: {loo_report['held_out_loss'].mean():.1f}")

Final production params (fit on all 8 GT dates): {'transit_center_s': 15.0, 'transit_spread_s': 20.0, 'rssi_weight': 5.0, 'min_match_score': 0.6}
In-sample total loss: 19692.0  |  mean held-out loss: 2781.2


## Apply to all 14 dual-sensor dates

Not just the 8 GT dates -- the calibrated matcher is applied to every
date where both sensors have data.

In [10]:
a_dates = set(sess_a["start_time"].dt.date)
b_dates = set(sess_b["start_time"].dt.date)
dual_dates = sorted(a_dates & b_dates)
print(f"{len(dual_dates)} dual-sensor dates: {dual_dates}")

all_matches = []
unmatched_rows = []
for date in dual_dates:
    ws = pd.Timestamp(date)
    we = ws + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    a_anchors = build_anchors_for_window(sess_a, ws, we)
    b_anchors = build_anchors_for_window(sess_b, ws, we)
    matches = match_travelers(a_anchors, b_anchors, final_params)
    for m in matches:
        m["date"] = date
    all_matches.extend(matches)

    matched_a = {m["traveler_id_a"] for m in matches}
    matched_b = {m["traveler_id_b"] for m in matches}
    for a in a_anchors:
        if a["traveler_id"] not in matched_a:
            unmatched_rows.append({"date": date, "sensor": "A", "traveler_id": a["traveler_id"]})
    for b in b_anchors:
        if b["traveler_id"] not in matched_b:
            unmatched_rows.append({"date": date, "sensor": "B", "traveler_id": b["traveler_id"]})

matched_df = pd.DataFrame(all_matches)
unmatched_df = pd.DataFrame(unmatched_rows)
print(f"Total matched pairs across all dual-sensor dates: {len(matched_df)}")
print(f"Total unmatched (single-sensor-only) travelers: {len(unmatched_df)}")

14 dual-sensor dates: [datetime.date(2026, 4, 11), datetime.date(2026, 5, 2), datetime.date(2026, 5, 3), datetime.date(2026, 5, 9), datetime.date(2026, 5, 16), datetime.date(2026, 5, 17), datetime.date(2026, 5, 18), datetime.date(2026, 5, 19), datetime.date(2026, 5, 20), datetime.date(2026, 5, 23), datetime.date(2026, 5, 24), datetime.date(2026, 5, 27), datetime.date(2026, 5, 28), datetime.date(2026, 5, 31)]


Total matched pairs across all dual-sensor dates: 6391
Total unmatched (single-sensor-only) travelers: 1192


## Validation: spot-check a few matched pairs

In [11]:
sample = matched_df.sort_values("match_score", ascending=False).head(5)
for _, row in sample.iterrows():
    print(f"{row['date']}  {row['traveler_id_a']} <-> {row['traveler_id_b']}  "
          f"direction={row['direction']}  gap={row['transit_gap_s']:.1f}s  score={row['match_score']:.2f}")

2026-05-16  A-97-5 <-> B-184-6  direction=A_to_B  gap=14.0s  score=83.50
2026-05-23  A-255-264 <-> B-405-78  direction=A_to_B  gap=18.0s  score=80.99
2026-05-24  A-307-14 <-> B-449-5  direction=B_to_A  gap=13302.0s  score=80.06
2026-05-16  A-101-294 <-> B-188-269  direction=A_to_B  gap=8.0s  score=77.71
2026-05-09  A-80-22 <-> B-166-29  direction=B_to_A  gap=27.0s  score=76.17


## Write output

In [12]:
out_cols = ["date", "traveler_id_a", "traveler_id_b", "direction", "transit_gap_s", "match_score"]
matched_out = matched_df[out_cols]
matched_out.to_csv(f"{DATA_DIR}/travelers_matched_direction.csv", index=False)
matched_out.to_parquet(f"{DATA_DIR}/travelers_matched_direction.parquet", index=False)
print(f"wrote {len(matched_out)} matched pairs to travelers_matched_direction.csv / .parquet")

wrote 6391 matched pairs to travelers_matched_direction.csv / .parquet
